In [1]:
import pandas as pd 

df = pd.read_csv("BWSC route_csv/80kmRoute.csv")

In [2]:
import numpy as np

dist = df["distance"].to_numpy()  # 누적거리 [m]
ele = df["ele"].to_numpy()        # 고도 [m]

ds = np.diff(dist)
dh = np.diff(ele)

slope_raw = dh / ds

def get_slope_profile(dist, ele, s0, v_ref, dt, N):
    ds = np.diff(dist)
    dh = np.diff(ele)
    slope_raw = dh / ds

    # 예측 horizon 동안의 대략적인 위치
    s_pred = s0 + np.arange(N) * v_ref * dt

    # 각 s_pred가 dist의 어느 구간에 있는지 찾기
    idx = np.searchsorted(dist[1:], s_pred)

    idx = np.clip(idx, 0, len(slope_raw) - 1)

    return slope_raw[idx]

/tmp/ipykernel_10672/1236132640.py:9: RuntimeWarning: invalid value encountered in divide
  slope_raw = dh / ds


In [ ]:
import cvxpy as cp
import numpy as np

N = 20          # 예측 horizon
dt = 1.0        # time step
m = 300.0       # vehicle mass

v = cp.Variable(N + 1)
s = cp.Variable(N + 1)
E = cp.Variable(N + 1)
u = cp.Variable(N)
u_bar = np.ones(N)*200.0

v0 = 60/3.6
s0 = 0.0
E0 = 5_000_000.0

v_ref = np.ones(N + 1) * 60/3.6
GHI = np.ones(N) * 800.0

qv = 10.0
qE = 1e-8
ru = 1e-3

constraints = [
    v[0] == v0,
    s[0] == s0,
    E[0] == E0
]

cost = 0
slope = get_slope_profile(dist, ele, s0, v_ref[0], dt, N)

for k in range(N):
    F_roll = 0.005 * m * 9.81
    F_slope = m * 9.81 * slope[k]

    # 단순화를 위해 공기저항은 기준속도 근처에서 상수처럼 처리
    # F_aero = 0.5 * 1.2 * 0.12 * 1.0 * (v_ref[k] ** 2)

    F_rest = F_roll + F_slope

    constraints += [
        v[k+1] == v[k]*(1-dt*0.5*1.2*0.12*v_ref[k]/m)
                  + (dt/m)*u[k]
                  - (dt/m)*F_rest,
        s[k+1] == s[k] + dt * v[k],
    ]

    # P_motor = u[k] * v_ref[k]
    #P_solar = 0.22 * 4.0 * GHI[k]

    constraints += [
        E[k+1] == E[k] - dt*(u_bar[k])*v[k] + -dt*(v_ref[k])*u[k] + dt*(0.22*4.0-1)
    ]

    constraints += [
        60/3.6 <= v[k],
        v[k] <= 100/3.6,
        0 <= u[k],
        u[k] <= 1000,
        E[k] >= 1_000_000
    ]

    cost += qv * cp.square(v[k] - v_ref[k])
    cost += qE * cp.square(E[k] - 3_000_000)
    cost += ru * cp.square(u[k])

problem = cp.Problem(cp.Minimize(cost), constraints)
problem.solve(solver=cp.OSQP)

print("optimal first control:", u.value[0])
print("predicted speed:", v.value * 3.6)
print("predicted energy:", E.value)

/tmp/ipykernel_10672/1236132640.py:14: RuntimeWarning: invalid value encountered in divide
  slope_raw = dh / ds


optimal first control: 235.86543842883682
predicted speed: [ 58.58955173  59.46947982  63.24497152  69.07121952  73.75360531
  76.08084517  79.46491261  84.20533527  87.39663627  87.54467287
  87.16388706  88.04930109  89.77342215  92.97278124  98.44103677
 106.80062304 117.542216   130.11520491 141.87694374 141.97672481
 141.78985151]
predicted energy: [5000000.13848982 4992841.65011214 4981402.93929841 4966014.67380091
 4951169.43737423 4939173.09736227 4926670.38997197 4911603.39628969
 4898372.39800575 4889989.26216952 4883423.0282474  4876127.58345041
 4868684.00872387 4861245.86307126 4851304.21227002 4837358.16879216
 4819486.2146721  4797169.53308168 4774830.33253568 4765803.7440524
 4757915.99635889]


/tmp/ipykernel_10672/3029386789.py:71: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  problem.solve(solver=cp.OSQP)


In [4]:
np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])[:, 2]

array([ 3,  6,  9, 12])